# mimic-video LIBERO Spatial-One Eval (10 Tasks x 10 Episodes)

This notebook runs LIBERO eval for **mimic-video / VAM** on the old `libero_spatial` benchmark: 10 spatial-one tasks with 10 episodes per task using CUDA device 3. It saves rollout videos without reward/value overlays and writes timestep-based model traces to JSONL, CSV, summary JSON, and a metrics dictionary markdown file.

The notebook uses the conda environment `mimic_video_eval` and cached LIBERO T5 embeddings.

In [ ]:
import os
import subprocess
from pathlib import Path

MIMIC_ROOT = Path('/home/motovilovil/Robotics/robotics_project/mimic-video')
MODEL_ROOT = MIMIC_ROOT / 'model'
LIBERO_EVAL_ROOT = MIMIC_ROOT / 'eval' / 'libero'
CONDA_ENV_ROOT = Path('/home/motovilovil/miniconda3/envs/mimic_video_eval')
PYTHON = CONDA_ENV_ROOT / 'bin' / 'python'

CHECKPOINT_DIR = MODEL_ROOT / 'checkpoints'
OUTPUT_ROOT = MIMIC_ROOT / 'eval_outputs' / 'libero_spatial_one_10tasks_10eps_device3_trace_modelonly'
ROLLOUT_DIR = OUTPUT_ROOT / 'videos'
METRICS_DIR = OUTPUT_ROOT / 'metrics'

ACTION_MODEL = CHECKPOINT_DIR / 'action_decoder' / 'w2a_libero_spatial_one_v2w_libero_spatial_agentview_lora_rank256_lr1.778e-04_bsz32_iter_000007540_fused_lr1.000e-04_layer20_bsz128_iter_000019998.pt'
VIDEO_MODEL = CHECKPOINT_DIR / 'video_backbone' / 'v2w_libero_spatial_agentview_lora_rank256_lr1.778e-04_bsz32_iter_000007540_fused.pt'
STATS = CHECKPOINT_DIR / 'dataset_statistics' / 'libero_spatial_one.json'
T5_EMBEDDINGS = Path('/home/motovilovil/.cache/huggingface/hub/models--nvidia--Cosmos-Policy-LIBERO-Predict2-2B/snapshots/cb689ec0e3347c13667d70a78a3447388f5c3bb8/libero_t5_embeddings.pkl')

for p in [ROLLOUT_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '3'
env['MUJOCO_GL'] = 'egl'
env['TOKENIZERS_PARALLELISM'] = 'false'
env['WANDB_MODE'] = 'disabled'
env['WANDB_SILENT'] = 'true'
env['MIMIC_VIDEO_DISABLE_WORLD_TQDM'] = '1'
env['LIBERO_CONFIG_PATH'] = str(MIMIC_ROOT / 'eval_outputs' / 'libero_config')
env['PYTHONPATH'] = f"{MODEL_ROOT}:{LIBERO_EVAL_ROOT / 'LIBERO'}:{env.get('PYTHONPATH', '')}"

print('Python:', PYTHON)
print('CUDA_VISIBLE_DEVICES:', env['CUDA_VISIBLE_DEVICES'])
print('Output:', OUTPUT_ROOT)
print('T5 embeddings:', T5_EMBEDDINGS)

## Ensure Checkpoints

If the minimal mimic-video LIBERO-spatial checkpoint files are missing, this cell downloads only the files needed for the smoke eval. The video/action weights are stored under `mimic-video/model/checkpoints/`.

In [ ]:
import libero
import shutil

# The local LIBERO checkout is first in PYTHONPATH, but it may miss benchmark asset files.
# Mirror the spatial benchmark assets from the conda package before running eval.
LIBERO_PACKAGE_ROOT = Path(libero.__file__).resolve().parent / 'libero'
LOCAL_LIBERO_ROOT = LIBERO_EVAL_ROOT / 'LIBERO' / 'libero' / 'libero'
for rel in ['init_files/libero_spatial', 'bddl_files/libero_spatial']:
    src = LIBERO_PACKAGE_ROOT / rel
    dst = LOCAL_LIBERO_ROOT / rel
    if src.exists() and (not dst.exists() or not any(dst.iterdir())):
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Copied {rel} from conda LIBERO package.')

required = [ACTION_MODEL, VIDEO_MODEL, STATS, T5_EMBEDDINGS]
missing = [p for p in required if not p.exists()]
if missing:
    print('Missing files:')
    for p in missing:
        print('  ', p)
    cmd = [str(PYTHON), str(LIBERO_EVAL_ROOT / 'download_spatial_one_checkpoints.py'), '--checkpoint-dir', str(CHECKPOINT_DIR)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(MIMIC_ROOT), env=env, check=True)
else:
    print('All required files are present.')

## Run LIBERO Spatial-One Eval On CUDA Device 3

This launches one `eval/libero/run.py` process on CUDA device 3. It uses `libero_spatial` and stops after 100 total episodes, which corresponds to the old 10 spatial-one tasks with 10 episodes per task. No reward/value text is drawn on the videos because mimic-video does not predict reward value.

In [ ]:
from pathlib import Path

PYTHON = Path('/home/motovilovil/miniconda3/envs/mimic_video_eval/bin/python')
if not PYTHON.exists():
    raise FileNotFoundError(f'Conda eval Python not found: {PYTHON}')

cmd = [
    str(PYTHON), 'run.py',
    '--vam_experiment_name', 'w2a_libero_spatial_one_v2w_libero_spatial_agentview_lora_rank256_lr1.778e-04_bsz32_iter_000007540_fused_lr1.000e-04_layer20_bsz128',
    '--vam_video_model_path', str(VIDEO_MODEL),
    '--vam_action_model_path', str(ACTION_MODEL),
    '--vam_dataset_statistics_path', str(STATS),
    '--vam_img_horizon', '5',
    '--vam_lowdim_horizon', '1',
    '--vam_stop_video_denoising_step', '0',
    '--vam_num_execute_actions', '5',
    '--task_suite_name', 'libero_spatial',
    '--num_trials_per_task', '10',
    '--max_eval_episodes', '100',
    '--rollout_dir', str(ROLLOUT_DIR),
    '--metrics_dir', str(METRICS_DIR),
    '--t5_embeddings_path', str(T5_EMBEDDINGS),
    '--seed', '0',
    '--no-use-cuda-graphs',
]
print('Single-process command template (not executed):')
print(' '.join(cmd))
print('Template only; the next cell launches the single-GPU eval on CUDA device 3.')

In [ ]:
import csv
import json
import shutil
from pathlib import Path

PYTHON = Path('/home/motovilovil/miniconda3/envs/mimic_video_eval/bin/python')
if not PYTHON.exists():
    raise FileNotFoundError(f'Conda eval Python not found: {PYTHON}')

base_cmd = [
    str(PYTHON), 'run.py',
    '--vam_experiment_name', 'w2a_libero_spatial_one_v2w_libero_spatial_agentview_lora_rank256_lr1.778e-04_bsz32_iter_000007540_fused_lr1.000e-04_layer20_bsz128',
    '--vam_video_model_path', str(VIDEO_MODEL),
    '--vam_action_model_path', str(ACTION_MODEL),
    '--vam_dataset_statistics_path', str(STATS),
    '--vam_img_horizon', '5',
    '--vam_lowdim_horizon', '1',
    '--vam_stop_video_denoising_step', '0',
    '--vam_num_execute_actions', '5',
    '--task_suite_name', 'libero_spatial',
    '--num_trials_per_task', '10',
    '--max_eval_episodes', '100',
    '--eval_world_size', '1',
    '--t5_embeddings_path', str(T5_EMBEDDINGS),
    '--seed', '0',
    '--no-use-cuda-graphs',
]

rank_specs = [(0, '3')]
processes = []
for rank, cuda_device in rank_specs:
    rank_root = OUTPUT_ROOT / f'rank{rank}'
    rank_rollouts = rank_root / 'videos'
    rank_metrics = rank_root / 'metrics'
    rank_rollouts.mkdir(parents=True, exist_ok=True)
    rank_metrics.mkdir(parents=True, exist_ok=True)

    rank_env = env.copy()
    rank_env['CUDA_VISIBLE_DEVICES'] = cuda_device
    cmd = base_cmd + [
        '--eval_rank', str(rank),
        '--rollout_dir', str(rank_rollouts),
        '--metrics_dir', str(rank_metrics),
    ]
    print(f'Launching eval on CUDA device {cuda_device}:')
    print(' '.join(cmd))
    processes.append(subprocess.Popen(cmd, cwd=str(LIBERO_EVAL_ROOT), env=rank_env))

for process in processes:
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)

# Merge rank-local trace outputs into OUTPUT_ROOT/videos and OUTPUT_ROOT/metrics.
ROLLOUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

episode_traces = []
chunk_rows = []
action_rows = []
for rank, _ in rank_specs:
    rank_root = OUTPUT_ROOT / f'rank{rank}'
    rank_metrics = rank_root / 'metrics'
    for video in (rank_root / 'videos').glob('*.mp4'):
        shutil.copy2(video, ROLLOUT_DIR / video.name)
    trace_path = rank_metrics / 'episode_traces.jsonl'
    if trace_path.exists():
        episode_traces.extend(json.loads(line) for line in trace_path.read_text().splitlines() if line.strip())
    chunk_path = rank_metrics / 'chunk_metrics.jsonl'
    if chunk_path.exists():
        chunk_rows.extend(json.loads(line) for line in chunk_path.read_text().splitlines() if line.strip())
    action_path = rank_metrics / 'action_metrics.jsonl'
    if action_path.exists():
        action_rows.extend(json.loads(line) for line in action_path.read_text().splitlines() if line.strip())

episode_traces.sort(key=lambda ep: int(ep['meta']['total_episode_idx']))
chunk_rows.sort(key=lambda row: (int(row['total_episode_idx']), int(row['inference_step_idx'])))
action_rows.sort(key=lambda row: (int(row['total_episode_idx']), int(row['action_index'])))

def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def write_csv(path, rows):
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    fieldnames = sorted({key for row in rows for key in row})
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows([{key: json.dumps(value) if isinstance(value, (list, dict)) else value for key, value in row.items()} for row in rows])

write_jsonl(METRICS_DIR / 'episode_traces.jsonl', episode_traces)
write_jsonl(METRICS_DIR / 'chunk_metrics.jsonl', chunk_rows)
write_jsonl(METRICS_DIR / 'action_metrics.jsonl', action_rows)
write_csv(METRICS_DIR / 'chunk_metrics.csv', chunk_rows)
write_csv(METRICS_DIR / 'action_metrics.csv', action_rows)

successes = sum(bool(ep['meta']['success']) for ep in episode_traces)
summary = {
    'num_episodes': len(episode_traces),
    'num_successes': successes,
    'success_rate': successes / len(episode_traces) if episode_traces else 0.0,
    'total_inference_steps': sum(int(ep['meta']['inference_step_count']) for ep in episode_traces),
    'total_chunks': len(chunk_rows),
    'total_actions': len(action_rows),
    'note': 'No episode-level action/model metric averages are computed; metrics are stored per chunk and per action.',
}
(METRICS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
metrics_md = OUTPUT_ROOT / 'rank0' / 'metrics' / 'METRICS.md'
if metrics_md.exists():
    shutil.copy2(metrics_md, METRICS_DIR / 'METRICS.md')

print({'episodes': len(episode_traces), 'chunks': len(chunk_rows), 'actions': len(action_rows), 'successes': successes})

## Inspect Outputs

In [ ]:
import json
import pandas as pd

videos = sorted(ROLLOUT_DIR.glob('*.mp4'))
print(f'Videos ({len(videos)}):')
for p in videos:
    print(f'  {p.name} ({p.stat().st_size / 1024:.1f} KiB)')

chunk_metrics_csv = METRICS_DIR / 'chunk_metrics.csv'
action_metrics_csv = METRICS_DIR / 'action_metrics.csv'
episode_traces_jsonl = METRICS_DIR / 'episode_traces.jsonl'
summary_json = METRICS_DIR / 'summary.json'
dictionary_md = METRICS_DIR / 'METRICS.md'

chunk_df = pd.read_csv(chunk_metrics_csv) if chunk_metrics_csv.exists() else pd.DataFrame()
action_df = pd.read_csv(action_metrics_csv) if action_metrics_csv.exists() else pd.DataFrame()
print('Chunk rows:', len(chunk_df), 'columns:', len(chunk_df.columns))
display(chunk_df.head())
print('Action rows:', len(action_df), 'columns:', len(action_df.columns))
display(action_df.head())

if episode_traces_jsonl.exists():
    first_trace = json.loads(episode_traces_jsonl.read_text().splitlines()[0])
    print('First trace meta:')
    print(json.dumps(first_trace['meta'], indent=2, ensure_ascii=False))
if summary_json.exists():
    summary = json.loads(summary_json.read_text())
    print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Metrics dictionary:', dictionary_md)